# P2: Learning-Paradigm Identifier — DistilBERT Fine-Tuning in Keras (U3 & U4)

This notebook trains a **DistilBERT** sequence classifier in **TensorFlow / Keras** on the 200 synthetic scenarios and evaluates on the 20 hand-labelled scenarios.

**Hardware accelerator:** Select `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
!pip install -q transformers tensorflow pandas numpy scikit-learn

In [ ]:
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from transformers import DistilBertTokenizerFast, TFDistilBertForSequenceClassification

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

### 1. Load Datasets
Upload `synthetic.jsonl` and `handlabelled_20.csv` to Colab's file browser or run the cell below.

In [ ]:
LABELS = ["supervised", "unsupervised", "reinforcement"]
l2i = {l: i for i, l in enumerate(LABELS)}
i2l = {i: l for i, l in enumerate(LABELS)}

# Load synthetic and hand-labelled data
syn = pd.read_json("synthetic.jsonl", lines=True)
hand = pd.read_csv("handlabelled_20.csv")

print(f"Loaded {len(syn)} synthetic scenarios and {len(hand)} hand-labelled scenarios.")
print(syn['label'].value_counts())

### 2. Tokenize with DistilBertTokenizerFast

In [ ]:
tok = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def encode(texts):
    return dict(tok(list(texts), truncation=True, padding=True, max_length=128, return_tensors="tf"))

X_train = encode(syn.text)
y_train = syn.label.map(l2i).values
X_test = encode(hand.text)

### 3. Build & Compile Keras TFDistilBert Model

In [ ]:
model = TFDistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=3
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

model.summary()

### 4. Train Model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=4,
    batch_size=16
)

### 5. Evaluate on Hand-Labelled 20 Scenarios

In [ ]:
logits = model.predict(X_test).logits
probs = tf.nn.softmax(logits, axis=1).numpy()

hand["pred_bert"] = [LABELS[i] for i in logits.argmax(axis=1)]
hand["bert_conf"] = probs.max(axis=1)
hand["is_correct"] = hand.pred_bert == hand.label

acc = hand["is_correct"].mean()
print(f"\nDistilBERT Accuracy on Hand-Labelled Test Set: {acc * 100:.2f}%")
print("\nPer-Domain Accuracy:")
print(hand.groupby("domain")["is_correct"].agg(["count", "mean"]))

# Display predictions
display(hand[["domain", "label", "pred_bert", "bert_conf", "is_correct"]])

### 6. Save Model and Download for App

In [ ]:
model.save_pretrained("distilbert_paradigm")
tok.save_pretrained("distilbert_paradigm")
!zip -r distilbert_paradigm.zip distilbert_paradigm

from google.colab import files
files.download("distilbert_paradigm.zip")
print("Downloaded distilbert_paradigm.zip!")